In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from itertools import combinations

In [15]:
HTYPE_DIR = "/Users/hridikpunukollu/Documents/Acads/MATH 7243/Project/renders/epidural"  # <-- change this

WINDOW_NAMES = [
    "brain_bone_window",
    "brain_window",
    "max_contrast_window",
    "subdural_window",
]
IMG_SIZE = (512, 512)
MAX_SAMPLES = 5000
SEED = 42

In [16]:
first_dir = os.path.join(HTYPE_DIR, WINDOW_NAMES[0])
all_files = [f for f in os.listdir(first_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))]

# Keep only images where all 4 windows exist
valid = []
for fname in all_files:
    paths = {}
    for w in WINDOW_NAMES:
        p = os.path.join(HTYPE_DIR, w, fname)
        if not os.path.isfile(p):
            break
        paths[w] = p
    else:
        valid.append({"filename": fname, "windows": paths})

print(f"Found {len(valid)} images with all 4 windows")

Found 1694 images with all 4 windows


In [17]:
rng = np.random.default_rng(SEED)
sample = valid if len(valid) <= MAX_SAMPLES else [valid[i] for i in rng.choice(len(valid), MAX_SAMPLES, replace=False)]

# shape: (n_samples, 4, H*W)
all_channels = {w: [] for w in WINDOW_NAMES}

for rec in sample:
    for w in WINDOW_NAMES:
        img = Image.open(rec["windows"][w]).convert("L").resize(IMG_SIZE)
        all_channels[w].append(np.asarray(img, dtype=np.float32).ravel())

for w in WINDOW_NAMES:
    all_channels[w] = np.stack(all_channels[w])  # (n_samples, H*W)

print(f"Loaded {len(sample)} images, each channel shape: {all_channels[WINDOW_NAMES[0]].shape}")

Loaded 1694 images, each channel shape: (1694, 262144)


In [18]:
pairs = list(combinations(WINDOW_NAMES, 2))
corr_results = {}

for w1, w2 in pairs:
    # per-image correlation, then average
    rs = [np.corrcoef(all_channels[w1][i], all_channels[w2][i])[0, 1] for i in range(len(sample))]
    corr_results[(w1, w2)] = np.nanmean(rs)

# Build symmetric matrix
corr_matrix = pd.DataFrame(np.eye(4), index=WINDOW_NAMES, columns=WINDOW_NAMES)
for (w1, w2), r in corr_results.items():
    corr_matrix.loc[w1, w2] = r
    corr_matrix.loc[w2, w1] = r

corr_matrix.round(4)

,brain_bone_window,brain_window,max_contrast_window,subdural_window
brain_bone_window,1.0000,0.8796,0.8875,0.8689
brain_window,0.8796,1.0000,0.8407,0.9457
max_contrast_window,0.8875,0.8407,1.0000,0.7514
subdural_window,0.8689,0.9457,0.7514,1.0000


In [19]:
avg_corr = corr_matrix.apply(lambda row: row.drop(row.name).mean(), axis=1).sort_values(ascending=False)

print("Mean correlation with other 3 channels:\n")
print(avg_corr.round(4).to_string())
print(f"\n→ Drop '{avg_corr.idxmax()}' (most redundant)")
print(f"→ Keep {[w for w in WINDOW_NAMES if w != avg_corr.idxmax()]}")

Mean correlation with other 3 channels:

brain_window           0.8887
brain_bone_window      0.8787
subdural_window        0.8553
max_contrast_window    0.8266

→ Drop 'brain_window' (most redundant)
→ Keep ['brain_bone_window', 'max_contrast_window', 'subdural_window']
